In [2]:
import os
import re
import json
import math
import argparse
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

In [3]:
# We try to import TensorFlow/Keras. If not installed, we skip DL gracefully.
use_dl = True
try:
    import tensorflow as tf
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Conv1D, MaxPooling1D
    from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
except Exception as e:
    print("[!] TensorFlow/Keras not available. Skipping deep learning models.")
    print("    Error:", e)
    use_dl = False


In [14]:
# End-to-end: data loading, cleaning, baselines, deep models, metrics, artifacts.
# Outputs are saved to /mnt/data/

# ------------------------- Paths & Args -------------------------
DEF_DATA = r"C:\Users\Mohammad\Desktop\Hands-On Projects\Fake News Detection\FA-KES-Dataset.csv"  # Path to FA-KES CSV 
OUT_DIR = Path(r"C:\Users\Mohammad\Desktop\Hands-On Projects\Fake News Detection\outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

parser = argparse.ArgumentParser(description="HW4 Q2 Fake News Detection on FA-KES")
parser.add_argument("--data", type=str, default=DEF_DATA, help="Path to FA-KES CSV")
parser.add_argument("--test_size", type=float, default=0.2, help="Test split size")
parser.add_argument("--seed", type=int, default=42, help="Random seed")
parser.add_argument("--max_features", type=int, default=50000, help="TF-IDF max features")
parser.add_argument("--max_vocab", type=int, default=40000, help="Tokenizer vocab size for DL")
parser.add_argument("--max_len", type=int, default=300, help="Sequence length for DL")
parser.add_argument("--emb_dim", type=int, default=100, help="Embedding dim for DL")
parser.add_argument("--epochs", type=int, default=8, help="Epochs for DL")
parser.add_argument("--batch_size", type=int, default=64, help="Batch size for DL")
parser.add_argument("--glove_path", type=str, default="", help="Optional path to GloVe txt (100d)")
args = parser.parse_args([]) if "__file__" not in globals() else parser.parse_args()

DATA_PATH = Path(args.data)


In [ ]:
# ------------------------- Utilities -------------------------
def load_csv_flex(path: Path) -> pd.DataFrame:
    encs = ["utf-8", "utf-8-sig", "latin1", "cp1252"]
    last_err = None
    for enc in encs:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"- Loaded CSV with encoding: {enc}")
            return df
        except Exception as e:
            last_err = e
    raise last_err


Data Analysis exploration (EDA)

In [27]:
df = load_csv_flex(DATA_PATH)
print(df['labels'].value_counts(normalize=True))
df

- Loaded CSV with encoding: latin1
labels
1    0.529851
0    0.470149
Name: proportion, dtype: float64


,unit_id,article_title,article_content,source,date,location,labels
0,1914947530,Syria attack symptoms consistent with nerve ag...,Wed 05 Apr 2017 Syria attack symptoms consiste...,nna,4/5/2017,idlib,0
1,1914947532,Homs governor says U.S. attack caused deaths b...,Fri 07 Apr 2017 at 0914 Homs governor says U.S...,nna,4/7/2017,homs,0
2,1914947533,Death toll from Aleppo bomb attack at least 112,Sun 16 Apr 2017 Death toll from Aleppo bomb at...,nna,4/16/2017,aleppo,0
3,1914947534,Aleppo bomb blast kills six Syrian state TV,Wed 19 Apr 2017 Aleppo bomb blast kills six Sy...,nna,4/19/2017,aleppo,0
4,1914947535,29 Syria Rebels Dead in Fighting for Key Alepp...,Sun 10 Jul 2016 29 Syria Rebels Dead in Fighti...,nna,7/10/2016,aleppo,0
...,...,...,...,...,...,...,...
799,1965511221,Turkish Bombardment Kills 20 Civilians in Syria,28-08-2016 Turkish Bombardment Kills 20 Civili...,manar,8/28/2016,aleppo,1
800,1965511222,Martyrs as Terrorists Shell Aleppos Salah Eddin,17-08-2016 Martyrs as Terrorists Shell Aleppos...,manar,8/1/2016,aleppo,1
801,1965511224,Chemical Attack Kills Five Syrians in Aleppo SANA,03-08-2016 Chemical Attack Kills Five Syrians ...,manar,8/3/2016,aleppo,0
802,1965511226,5 Killed as Russian Military Chopper Shot down...,01-08-2016 5 Killed as Russian Military Choppe...,manar,8/1/2016,idlib,1


In [ ]:
def format_date_abbrev(val):
    """ Convert the dateformated time into string as its presented in the the content """
    ts = pd.to_datetime(val, errors='coerce')
    if pd.isna(ts):
        parsed = None
        for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%m/%d/%Y", "%Y/%m/%d"):
            try:
                parsed = datetime.strptime(str(val), fmt)
                break
            except Exception:
                continue
        if parsed is None:
            return None
        return f"{parsed.strftime('%a').upper()}, {parsed.strftime('%b %d %Y')}"
    return f"{ts.strftime('%a')} {ts.strftime('%d %b %Y')}"

def infer_columns(df: pd.DataFrame):
    """Infer label and text columns from common FA-KES schema."""
    # label
    label_col = None
    for c in df.columns:
        if c.lower() in ["labels", "label", "class", "target", "y", "is_fake", "fake_label"]:
            label_col = c
            break
    # If not found, try binary numeric col
    if label_col is None:
        for c in df.columns:
            vals = df[c].dropna().unique()
            if len(set(vals)) == 2:
                # attempt simple mapping
                label_col = c
                break

    # text
    text_col = None
    for c in df.columns:
        if c.lower() in ["article_content", "text", "content", "body", "article"]:
            text_col = c
            break
    if text_col is None:
        # fallback: concatenate all object columns
        obj_cols = [c for c in df.columns if df[c].dtype == object]
        if not obj_cols:
            raise ValueError("No obvious text columns found.")
        text_col = None  # will handle later
    return text_col, label_col
def convert_date_time_to_string(val):
    formated_val = '%A, %B %d, %Y'
    RETURNED_DATE = val.strftime(formated_val)
    return RETURNED_DATE
def remove_date_from_text(row, text_col_name):
    txt = str(row[text_col_name])
    dstr = row.get('date_str')
    if dstr is None or not isinstance(dstr, str) or dstr.strip() == '':
        return txt
    # remove exact match and a no-comma variant, then collapse spaces
    txt = txt.replace(dstr, '')
    txt = txt.replace(dstr.replace(',', ''), '')
    return ' '.join(txt.split()).strip()

def basic_clean(text: str) -> str:
    text = str(text)
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
    text = re.sub(r"\b\d{1,3}(?:\.\d{1,3}){3}\b", " ", text)  # IPs
    text = re.sub(r"[^A-Za-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

def binarize_labels(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip().str.lower()
    mapping = {"real": 1, "fake": 0, "1": 1, "0": 0, "true": 1, "false": 0}
    if not set(s.unique()).issubset(set(mapping.keys())):
        # try to coerce numeric 0/1
        try:
            s = s.astype(float).astype(int)
            return s
        except:
            raise ValueError("Label values are not in {fake/real/0/1/true/false}.")
    return s.map(mapping).astype(int)

def print_and_save_report(name: str, y_true, y_pred, out_dir: Path):
    acc = accuracy_score(y_true, y_pred)
    pr, rc, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", pos_label=1, zero_division=0)
    print(f"\n=== {name} ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {pr:.4f}")
    print(f"Recall:    {rc:.4f}")
    print(f"F1:        {f1:.4f}")
    print(classification_report(y_true, y_pred, digits=3))
    cm = confusion_matrix(y_true, y_pred)
    print("Confusion Matrix:\n", cm)

    metrics = {
        "model": name,
        "accuracy": float(acc),
        "precision": float(pr),
        "recall": float(rc),
        "f1": float(f1),
        "confusion_matrix": cm.tolist(),
    }
    out_json = out_dir / f"{name.replace(' ', '_')}_metrics.json"
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)
    print(f"- Saved metrics to: {out_json}")


In [72]:
# ------------------------- Load & Prepare Data -------------------------
text_col, label_col = infer_columns(df)
print(f"- Detected text_col={text_col}, label_col={label_col}")

y_raw = df[label_col]
y = binarize_labels(y_raw)
df['date_str'] = df['date'].apply(format_date_abbrev)
df[text_col] = df.apply(lambda r: remove_date_from_text(r, text_col), axis=1) # for better usage I updated the current Cell as well. better to add new one.
# Compose text if needed
if text_col is None:
    obj_cols = [c for c in df.columns if df[c].dtype == object]
    X_raw = df[obj_cols].astype(str).apply(lambda r: " ".join(r.values), axis=1)
else:
    X_raw = df[text_col].fillna("")

X_clean = X_raw.apply(basic_clean)


- Detected text_col=article_content, label_col=labels


In [71]:
X_clean.head()

0    syria attack symptoms consistent with nerve ag...
1    at homs governor says u s attack caused deaths...
2    death toll from aleppo bomb attack at least th...
3    aleppo bomb blast kills six syrian state tv a ...
4    syria rebels dead in fighting for key aleppo r...
Name: article_content, dtype: object

In [73]:
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y, test_size=args.test_size, random_state=args.seed, stratify=y
)

print(f"- Train size: {len(X_train)} | Test size: {len(X_test)}")

# ------------------------- Classical Baselines -------------------------
pipelines = {
    "TFIDF + LinearSVC": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=args.max_features, ngram_range=(1,2), lowercase=True)),
        ("clf", LinearSVC())
    ]),
    "TFIDF + LogReg (liblinear)": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=args.max_features, ngram_range=(1,2), lowercase=True)),
        ("clf", LogisticRegression(max_iter=500, solver="liblinear"))
    ]),
}

baseline_table = []
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    print_and_save_report(name, y_test, preds, OUT_DIR)
    acc = accuracy_score(y_test, preds)
    pr, rc, f1, _ = precision_recall_fscore_support(y_test, preds, average="binary", pos_label=1, zero_division=0)
    baseline_table.append([name, acc, pr, rc, f1])

baseline_df = pd.DataFrame(baseline_table, columns=["Model","Accuracy","Precision","Recall","F1"]).sort_values("F1", ascending=False)
csv_out = OUT_DIR / "Baseline_Results.csv"
baseline_df.to_csv(csv_out, index=False)
print(f"- Baseline table saved to: {csv_out}")
print(baseline_df.round(4))


- Train size: 643 | Test size: 161

=== TFIDF + LinearSVC ===
Accuracy:  0.5217
Precision: 0.5465
Recall:    0.5529
F1:        0.5497
              precision    recall  f1-score   support

           0      0.493     0.487     0.490        76
           1      0.547     0.553     0.550        85

    accuracy                          0.522       161
   macro avg      0.520     0.520     0.520       161
weighted avg      0.521     0.522     0.522       161

Confusion Matrix:
 [[37 39]
 [38 47]]
- Saved metrics to: C:\Users\Mohammad\Desktop\Hands-On Projects\Fake News Detection\outputs\TFIDF_+_LinearSVC_metrics.json

=== TFIDF + LogReg (liblinear) ===
Accuracy:  0.5342
Precision: 0.5463
Recall:    0.6941
F1:        0.6114
              precision    recall  f1-score   support

           0      0.509     0.355     0.419        76
           1      0.546     0.694     0.611        85

    accuracy                          0.534       161
   macro avg      0.528     0.525     0.515       16

In [74]:
# ------------------------- Deep Learning Models -------------------------
def maybe_load_glove(glove_path: str, tok: Tokenizer, emb_dim: int, vocab_size: int):
    if not glove_path:
        return None
    glove_file = Path(glove_path)
    if not glove_file.exists():
        print(f"[!] GloVe not found at {glove_file}. Continuing with random init.")
        return None
    print(f"[i] Loading GloVe from {glove_file} ...")
    embeddings_index = {}
    with open(glove_file, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip().split(" ")
            word = parts[0]
            vec = np.asarray(parts[1:], dtype="float32")
            if vec.shape[0] == emb_dim:
                embeddings_index[word] = vec
    print(f"[i] GloVe entries: {len(embeddings_index):,}")

    embedding_matrix = np.random.normal(scale=0.02, size=(vocab_size, emb_dim)).astype(np.float32)
    for word, i in tok.word_index.items():
        if i >= vocab_size:
            continue
        vec = embeddings_index.get(word)
        if vec is not None:
            embedding_matrix[i] = vec
    return embedding_matrix

if use_dl:
    MAX_VOCAB = args.max_vocab
    MAX_LEN = args.max_len
    EMB_DIM = args.emb_dim

    tok = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
    tok.fit_on_texts(X_train.tolist())
    Xtr_seq = tok.texts_to_sequences(X_train.tolist())
    Xte_seq = tok.texts_to_sequences(X_test.tolist())
    Xtr = pad_sequences(Xtr_seq, maxlen=MAX_LEN, padding="post", truncating="post")
    Xte = pad_sequences(Xte_seq, maxlen=MAX_LEN, padding="post", truncating="post")

    vocab_size = min(MAX_VOCAB, len(tok.word_index)+1)
    emb_matrix = maybe_load_glove(args.glove_path, tok, EMB_DIM, vocab_size)

    def build_rnn_model():
        model = Sequential()
        model.add(Embedding(input_dim=vocab_size, output_dim=EMB_DIM, input_length=MAX_LEN,
                            weights=[emb_matrix] if emb_matrix is not None else None,
                            trainable=True if emb_matrix is not None else True))
        model.add(LSTM(64))
        model.add(Dropout(0.3))
        model.add(Dense(1, activation="sigmoid"))
        model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
        return model

    def build_hybrid_model():
        model = Sequential()
        model.add(Embedding(input_dim=vocab_size, output_dim=EMB_DIM, input_length=MAX_LEN,
                            weights=[emb_matrix] if emb_matrix is not None else None,
                            trainable=True if emb_matrix is not None else True))
        model.add(Conv1D(filters=128, kernel_size=5, activation="relu"))
        model.add(MaxPooling1D(pool_size=2))
        model.add(LSTM(64))
        model.add(Dropout(0.3))
        model.add(Dense(1, activation="sigmoid"))
        model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
        return model

    es = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
    
    # ----- RNN (LSTM) -----
    print("\n-- Training RNN (LSTM) ...")
    rnn_model = build_rnn_model()
    rnn_ckpt = OUT_DIR / "rnn_best.h5"
    mc_rnn = ModelCheckpoint(rnn_ckpt.as_posix(), monitor="val_loss", save_best_only=True, verbose=1)
    hist_rnn = rnn_model.fit(
        Xtr, y_train,
        validation_split=0.1,
        epochs=args.epochs,
        batch_size=args.batch_size,
        callbacks=[es, mc_rnn],
        verbose=1
    )
    # Evaluate
    rnn_pred = (rnn_model.predict(Xte) > 0.5).astype(int).ravel()
    print_and_save_report("RNN_LSTM", y_test, rnn_pred, OUT_DIR)

    # ----- Hybrid CNN→MaxPool→LSTM -----
    print("\n-- Training Hybrid CNN→LSTM ...")
    hyb_model = build_hybrid_model()
    hyb_ckpt = OUT_DIR / "hybrid_best.h5"
    mc_hyb = ModelCheckpoint(hyb_ckpt.as_posix(), monitor="val_loss", save_best_only=True, verbose=1)
    hist_hyb = hyb_model.fit(
        Xtr, y_train,
        validation_split=0.1,
        epochs=args.epochs,
        batch_size=args.batch_size,
        callbacks=[es, mc_hyb],
        verbose=1
    )
    hyb_pred = (hyb_model.predict(Xte) > 0.5).astype(int).ravel()
    print_and_save_report("Hybrid_CNN_LSTM", y_test, hyb_pred, OUT_DIR)

    # ----- Save training histories & simple plots -----
    try:
        import matplotlib.pyplot as plt

        def save_history_plot(hist, title, fname_prefix):
            acc = hist.history.get("accuracy", [])
            val_acc = hist.history.get("val_accuracy", [])
            loss = hist.history.get("loss", [])
            val_loss = hist.history.get("val_loss", [])

            plt.figure()
            plt.plot(acc, label="train_acc")
            plt.plot(val_acc, label="val_acc")
            plt.xlabel("Epoch")
            plt.ylabel("Accuracy")
            plt.title(f"{title} Accuracy")
            plt.legend()
            plt.savefig(OUT_DIR / f"{fname_prefix}_acc.png", bbox_inches="tight")
            plt.close()

            plt.figure()
            plt.plot(loss, label="train_loss")
            plt.plot(val_loss, label="val_loss")
            plt.xlabel("Epoch")
            plt.ylabel("Loss")
            plt.title(f"{title} Loss")
            plt.legend()
            plt.savefig(OUT_DIR / f"{fname_prefix}_loss.png", bbox_inches="tight")
            plt.close()

        save_history_plot(hist_rnn, "RNN (LSTM)", "rnn")
        save_history_plot(hist_hyb, "Hybrid CNN-LSTM", "hyb")
        print("-- Saved training curves to Output Folder")
    except Exception as e:
        print("!!! Could not plot histories:", e)

print("\n-- DONE. Artifacts in Output Folder")
# Files produced:
# - Baseline_Results.csv
# - <model>_metrics.json for each model
# - rnn_best.h5, hybrid_best.h5 (if DL ran)
# - rnn_acc.png, rnn_loss.png, hyb_acc.png, hyb_loss.png (if plotting succeeded)



-- Training RNN (LSTM) ...
Epoch 1/8


C:\Users\Mohammad\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - accuracy: 0.5373 - loss: 0.6924
Epoch 1: val_loss improved from None to 0.71080, saving model to C:/Users/Mohammad/Desktop/Hands-On Projects/Fake News Detection/outputs/rnn_best.h5


10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 157ms/step - accuracy: 0.5398 - loss: 0.6924 - val_accuracy: 0.4615 - val_loss: 0.7108
Epoch 2/8
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step - accuracy: 0.5456 - loss: 0.6879
Epoch 2: val_loss improved from 0.71080 to 0.70159, saving model to C:/Users/Mohammad/Desktop/Hands-On Projects/Fake News Detection/outputs/rnn_best.h5


10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 148ms/step - accuracy: 0.5381 - loss: 0.6876 - val_accuracy: 0.4615 - val_loss: 0.7016
Epoch 3/8
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - accuracy: 0.5670 - loss: 0.6861
Epoch 3: val_loss improved from 0.70159 to 0.69738, saving model to C:/Users/Mohammad/Desktop/Hands-On Projects/Fake News Detection/outputs/rnn_best.h5


...

C:\Users\Mohammad\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


 9/10 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.5034 - loss: 0.6927
Epoch 1: val_loss improved from None to 0.71502, saving model to C:/Users/Mohammad/Desktop/Hands-On Projects/Fake News Detection/outputs/hybrid_best.h5


10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 140ms/step - accuracy: 0.5087 - loss: 0.6936 - val_accuracy: 0.4615 - val_loss: 0.7150
Epoch 2/8
 9/10 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - accuracy: 0.5554 - loss: 0.6955
Epoch 2: val_loss did not improve from 0.71502
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 99ms/step - accuracy: 0.5657 - loss: 0.6884 - val_accuracy: 0.4615 - val_loss: 0.7255
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step

=== Hybrid_CNN_LSTM ===
Accuracy:  0.5280
Precision: 0.5280
Recall:    1.0000
F1:        0.6911
              precision    recall  f1-score   support

           0      0.000     0.000     0.000        76
           1      0.528     1.000     0.691        85

    accuracy                          0.528       161
   macro avg      0.264     0.500     0.346       161
weighted avg      0.279     0.528     0.365       161

Confusion Matrix:
 [[ 0 76]
 [ 0 85]]
- Saved metrics to: C:\Users\Mohammad\Desktop\Hands-On Projects\Fake News Detection\outputs\Hybrid_CNN_LSTM_metrics.json


C:\Users\Mohammad\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Mohammad\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Mohammad\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capita

-- Saved training curves to Output Folder

-- DONE. Artifacts in Output Folder
